# PA2: 01_sft

In [ ]:
import base64, io, zipfile, os, subprocess
from pathlib import Path
root = Path('/kaggle/working/pa2')
root.mkdir(parents=True, exist_ok=True)
zipfile.ZipFile(io.BytesIO(base64.b64decode('...'))).extractall(root)
os.chdir(root)
subprocess.run(['python', '-m', 'pip', 'uninstall', '-y', '-q', 'torchao'], check=False)
subprocess.run(['python', '-m', 'pip', 'install', '-q', '-r', 'requirements.txt'], check=True)
import torch, transformers, datasets
print({'torch': torch.__version__, 'transformers': transformers.__version__, 'datasets': datasets.__version__})
subprocess.run(['nvidia-smi'], check=True)
assert torch.cuda.device_count() == 2, 'Select GPU T4 x2'
assert all('T4' in torch.cuda.get_device_name(i) for i in range(2)), 'Expected two T4 GPUs'
subprocess.run(['python', '-m', 'pytest', '-q', 'tests'], check=True)


In [ ]:
subprocess.run(['python', '-m', 'scripts.prepare_data', '--tracks', 'sft'], check=True)

In [ ]:
import json
for split, count in [('train', 64), ('validation', 16)]:
    rows = json.loads(Path(f'outputs/data/sft_{split}.json').read_text())
    Path(f'outputs/data/pilot_{split}.json').write_text(json.dumps(rows[:count]))
subprocess.run(['python', '-m', 'scripts.train', '--task', 'sft', '--train', 'outputs/data/pilot_train.json', '--validation', 'outputs/data/pilot_validation.json', '--output', 'outputs/sft_pilot', '--max-steps', '2'], check=True)


In [ ]:
subprocess.run(['python', '-m', 'scripts.train', '--task', 'sft', '--train', 'outputs/data/sft_train.json', '--validation', 'outputs/data/sft_validation.json', '--output', 'outputs/sft'], check=True)

In [ ]:
print('Save this notebook version to preserve /kaggle/working/pa2/outputs.')